# ПЗ-6. Grounding и IoU

## 1. Цель
Проверить локализацию объекта по описанию на 200 примерах RefCOCO val.

## 2. Датасеты и модели
[RefCOCO UNC validation](https://huggingface.co/datasets/jxu124/refcoco/tree/9fba8200c5326e996f789191f095bd464ef1d09e): из 3811 референтов выбираются 200 различных изображений для оценки и ещё 5 для проверки координат. Поле split должно быть val. Текст берётся из исходных sentences, а COCO xywh — из raw_anns; это проверяется по сохранённым идентификаторам image_id и ann_id. Загружаются чистые оригиналы COCO train2014 без нарисованных рамок.

Используется один фиксированный UNC split. Его размеры сверены с [документацией TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/ref_coco). Смешение вариантов разбиения не применяется.

[Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), fp16. [Разъяснение разработчика о координатах](https://github.com/QwenLM/Qwen3-VL/issues/830): абсолютные пиксели изображения после resize. Это поведение именно Qwen2.5-VL.

Источник задания: `Практика_06_11_2026_1_week.docx`. Приложения А.1–А.7 отсутствуют; ниже приведены стандартные реализации. LoRA-проект не входит в эти шесть работ.

## 3. Подготовка окружения
Откройте этот notebook в Google Colab. Для ПЗ-1 выберите CPU, для ПЗ-3–6 — T4; для ПЗ-2 GPU ускорит получение признаков. Выполните установку, перезапустите сеанс, затем выполните остальные ячейки сверху вниз. Все результаты сохраняются в папке задания; скачайте её вместе с ноутбуком. Данные и веса кэшируются в соседней папке `data`. При переносе в Colab прежние результаты доступны только после загрузки файлов. GPU использует fp16. Числа в сохранённых выводах относятся только к реально выполненным ячейкам. Ячейки без execution_count не запускались.

In [ ]:
%pip install torch==2.9.1 torchvision==0.24.1 transformers==4.57.6 numpy==2.3.5 pandas==2.3.3 matplotlib==3.10.8 pillow==12.0.0 datasets==4.4.1 accelerate==1.12.0 sentencepiece==0.2.1 scikit-learn==1.7.2 pycocoevalcap==1.2 reportlab==4.4.5 huggingface-hub==0.36.2 safetensors==0.8.0 tokenizers==0.22.2 protobuf==6.33.2 ftfy==6.3.1 requests==2.34.2 fsspec==2025.10.0 pyarrow==26.0.1

In [1]:
LAB = 'PZ_6'
REVISIONS = {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'jxu124/refcoco': '9fba8200c5326e996f789191f095bd464ef1d09e'}

import os
import sys
import json
import random
import time
import math
import gc
import hashlib
import platform
import importlib.metadata
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd() if Path.cwd().name == LAB else Path.cwd() / LAB
ROOT.mkdir(exist_ok=True)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
DATA = ROOT.parent / "data"
DATA.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(DATA / "huggingface")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

def save_json(name, value):
    path = OUT / name
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    for attempt in range(20):
        try:
            temporary.replace(path)
            return
        except PermissionError:
            if attempt==19:
                raise
            time.sleep(0.1)

def save_figure(name):
    plt.savefig(OUT / name, dpi=130, bbox_inches="tight")
    plt.show()

versions = {p.metadata["Name"]: p.version for p in importlib.metadata.distributions() if p.metadata["Name"]}
environment = {"python": sys.version, "platform": platform.platform(), "seed": SEED,
               "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
               "versions": versions, "revisions": REVISIONS}
save_json("environment.json", environment)
print({k: v for k, v in environment.items() if k != "versions"})

C:\Users\darri\PycharmProjects\cv061126\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'seed': 42, 'device': 'cpu', 'gpu': None, 'revisions': {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'jxu124/refcoco': '9fba8200c5326e996f789191f095bd464ef1d09e'}}


In [2]:
from xml.sax.saxutils import escape
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as PDFImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from matplotlib import font_manager

def write_report(title, goal, work, findings, table=None, figure=None):
    pdfmetrics.registerFont(TTFont("Student", font_manager.findfont("DejaVu Sans")))
    style = ParagraphStyle("Student", fontName="Student", fontSize=10, leading=14, spaceAfter=8)
    heading = ParagraphStyle("Heading", parent=style, fontSize=14, leading=18, spaceAfter=12)
    story = [Paragraph(escape(title), heading), Paragraph(escape("Цель. " + goal), style),
             Paragraph(escape("Что было сделано. " + work), style)]
    for text in findings:
        story.append(Paragraph(escape(text), style))
    if table is not None and len(table):
        rows = [[Paragraph(escape(str(x)), style) for x in table.columns]]
        rows += [[Paragraph(escape(f"{x:.4f}" if isinstance(x, float) else str(x)), style) for x in row]
                 for row in table.itertuples(index=False, name=None)]
        item = Table(rows, repeatRows=1, colWidths=[495 / len(table.columns)] * len(table.columns))
        item.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "TOP"),
                                 ("GRID", (0,0), (-1,-1), 0.3, colors.grey)]))
        story.extend([item, Spacer(1, 10)])
    if figure is not None and Path(figure).exists():
        with Image.open(figure) as im:
            ratio = im.height / im.width
        story.append(PDFImage(str(figure), width=460, height=min(250, 460 * ratio)))
    story.append(Paragraph(escape("Индивидуальный вклад. Подготовлены реализации, протоколы экспериментов, тесты и оформление. Код и текст подготовлены с помощью ИИ-ассистента; фактические запуски и ограничения перечислены выше. Авторство невыполненных опытов не заявляется."), style))
    story.append(Paragraph(escape("Воспроизведение: notebook " + LAB + ".ipynb; seed=42. Версии среды и точные ревизии весов сохранены в results/environment.json. Контрольные вопросы разобраны в ноутбуке."), style))
    SimpleDocTemplate(str(ROOT / "report.pdf"), pagesize=(595,842), rightMargin=50, leftMargin=50,
                      topMargin=45, bottomMargin=45).build(story)
    print("PDF:", ROOT / "report.pdf")

In [3]:
import requests
from zipfile import ZipFile

def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(4):
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with temporary.open("wb") as stream:
                    for block in response.iter_content(1024 * 1024):
                        stream.write(block)
            temporary.replace(destination)
            return destination
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(attempt + 1)



def coco_image(record, year=2017):
    split = record["split"]
    image_id = record["image_id"]
    name = f"{image_id:012d}.jpg" if year == 2017 else f"COCO_{split}{year}_{image_id:012d}.jpg"
    path = download(f"https://s3.amazonaws.com/images.cocodataset.org/{split}{year}/{name}", DATA / f"{split}{year}" / name)
    with Image.open(path) as image:
        return image.convert("RGB")


## 4. Реализация
Предсказание ожидается как JSON с bbox_2d=[x1,y1,x2,y2]. Координаты переводятся по отдельным коэффициентам W_original/W_processed и H_original/H_processed. NaN, вырожденные и выходящие за границы рамки считаются невалидными. Координаты не переставляются и не обрезаются молча. Невалидные ответы входят в знаменатель метрик с IoU=0.

In [4]:
import re
from matplotlib.patches import Rectangle

def iou(a,b):
    if a is None or b is None:
        return 0.
    a,b = np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    if a.shape!=(4,) or b.shape!=(4,) or not np.isfinite(a).all() or not np.isfinite(b).all():
        return 0.
    widths_a, widths_b = np.maximum(a[2:]-a[:2],0),np.maximum(b[2:]-b[:2],0)
    intersection = np.maximum(np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2]),0).prod()
    union = widths_a.prod()+widths_b.prod()-intersection
    return float(intersection/union) if union>0 else 0.

def parse_box(text):
    cleaned = re.sub(r"```(?:json)?|```","",text).strip()
    decoder = json.JSONDecoder()
    value = None
    for match in re.finditer(r"[\[{]",cleaned):
        try:
            value,_ = decoder.raw_decode(cleaned[match.start():])
            break
        except json.JSONDecodeError:
            continue
    if isinstance(value,list) and len(value)==1 and isinstance(value[0],dict):
        value = value[0]
    if isinstance(value,dict):
        value = value.get("bbox_2d",value.get("bbox"))
    if not isinstance(value,list) or len(value)!=4:
        return None,"json_or_shape"
    if any(isinstance(v,bool) or not isinstance(v,(int,float)) for v in value):
        return None,"non_numeric"
    box = np.array(value,dtype=float)
    if not np.isfinite(box).all():
        return None,"non_finite"
    if box[2]<=box[0] or box[3]<=box[1]:
        return None,"degenerate"
    return box.tolist(),None

def original_box(box,original_size,processed_size):
    if box is None:
        return None,"parse_error"
    ow,oh = original_size
    pw,ph = processed_size
    x1,y1,x2,y2 = box
    if min(ow,oh,pw,ph)<=0 or min(x1,y1)<0 or x2>pw or y2>ph:
        return None,"out_of_bounds"
    return [x1*ow/pw,y1*oh/ph,x2*ow/pw,y2*oh/ph],None

def draw_box(ax,box,color,label):
    if box is not None:
        x1,y1,x2,y2 = box
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,linewidth=2,label=label))

In [5]:
assert iou([0,0,10,10],[0,0,10,10])==1.
assert iou([0,0,10,10],[10,10,20,20])==0.
assert abs(iou([0,0,10,10],[5,5,15,15])-1/7)<1e-12
assert iou([0,0,0,0],[0,0,10,10])==0.
assert iou(None,[0,0,10,10])==0.
assert iou([0,0,10,10],[0,0,5,5])==0.25
assert parse_box('```json\n{"bbox_2d":[1,2,20,30]}\n```')[0]==[1,2,20,30]
assert parse_box('[{"bbox_2d":[1,2,20,30]}]')[0]==[1,2,20,30]
assert parse_box('{"bbox_2d":[20,2,1,30]}')[0] is None
assert parse_box('{"bbox_2d":[true,2,20,30]}')[0] is None
assert parse_box('not JSON')[0] is None
converted,error = original_box([14,28,140,280],(640,480),(448,336))
assert np.allclose(converted,[20,40,200,400]) and error is None
assert original_box([0,0,449,336],(640,480),(448,336))[0] is None
anisotropic, error = original_box([14,28,140,280],(640,480),(448,280))
assert np.allclose(anisotropic,[20,48,200,480]) and error is None
assert parse_box('{"bbox_2d":[0,0,NaN,10]}')[0] is None
rng = np.random.default_rng(SEED)
for _ in range(100):
    a,b = rng.uniform(0,100,(2,4))
    a[2:] += a[:2]
    b[2:] += b[:2]
    assert 0<=iou(a,b)<=1 and abs(iou(a,b)-iou(b,a))<1e-12
save_json("unit_tests.json",{"IoU_identical":1.,"IoU_disjoint":0.,"IoU_overlap":iou([0,0,10,10],[5,5,15,15]),
                             "coordinate_result":converted,"anisotropic_result":anisotropic,"random_symmetry_cases":100,"parser":True,"passed":True})
print("IoU, JSON parsing and coordinate conversion tests passed")

IoU, JSON parsing and coordinate conversion tests passed


In [6]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

def load_vlm(max_pixels=1000000):
    if DEVICE.type != "cuda":
        raise RuntimeError("Для полного эксперимента требуется GPU T4 16 GB или совместимая CUDA GPU. Размер выборки не уменьшен.")
    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    processor = AutoProcessor.from_pretrained(model_id, revision=REVISIONS[model_id],
                                             min_pixels=4*28*28, max_pixels=max_pixels, use_fast=False)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, revision=REVISIONS[model_id], torch_dtype=torch.float16,
        attn_implementation="sdpa").to(DEVICE).eval()
    model.requires_grad_(False)
    return model, processor

def vlm_answer(model, processor, image, prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = processor(text=[chat], images=[image], return_tensors="pt",
                      images_kwargs={"min_pixels":processor.image_processor.min_pixels,
                                     "max_pixels":processor.image_processor.max_pixels})
    grid = batch["image_grid_thw"][0].tolist()
    n_visual = int(np.prod(grid) // processor.image_processor.merge_size**2)
    batch = batch.to(DEVICE)
    batch["pixel_values"] = batch["pixel_values"].to(torch.float16)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**batch, do_sample=False, max_new_tokens=max_new_tokens,
                                   use_cache=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    text = processor.batch_decode(generated[:, batch.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    return {"answer": text, "visual_tokens": n_visual, "seconds": elapsed,
            "grid_thw": grid, "processed_height": grid[1]*14, "processed_width": grid[2]*14,
            "generated_tokens": generated.shape[1]-batch.input_ids.shape[1]}

## 5. Эксперименты
Пять калибровочных изображений не входят в итоговые 200. На них проверяется документированный формат; он фиксируется до основной оценки. Нельзя выбирать масштаб отдельно для каждого тестового примера по максимальному IoU с эталоном.

In [7]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_url

dataset_id = "jxu124/refcoco"
url = hf_hub_url(dataset_id,"data/validation-00000-of-00001-bfeafdc84ca37aa2.parquet",
                 repo_type="dataset",revision=REVISIONS[dataset_id])
metadata_path = download(url,DATA/"refcoco_unc_validation.parquet")
metadata_sha = hashlib.sha256(metadata_path.read_bytes()).hexdigest()
assert metadata_sha=="df03b1b16873f92727f3df010afb3c7260e8396a86592d3c4fee053daca08c05"
metadata = pq.read_table(metadata_path).to_pylist()
assert len(metadata)==3811 and all(row["split"]=="val" for row in metadata)
assert len({row["image_id"] for row in metadata})==1500
order = np.random.default_rng(SEED).permutation(len(metadata)).tolist()
selected, seen_images = [], set()
for index in order:
    row = metadata[index]
    image_id = int(row["image_id"])
    if image_id in seen_images:
        continue
    seen_images.add(image_id)
    annotation = json.loads(row["raw_anns"])
    image_info = json.loads(row["raw_image_info"])
    assert annotation["id"]==row["ann_id"] and annotation["image_id"]==image_id
    assert image_info["id"]==image_id
    expressions = [sentence["raw"] for sentence in row["sentences"]]
    assert expressions
    x,y,w,h = [float(v) for v in annotation["bbox"]]
    assert w>0 and h>0
    target_box = [x,y,x+w,y+h]
    assert np.allclose(target_box,row["bbox"])
    selected.append({"question_id":str(row["sent_ids"][0]),"ref_id":row["ref_id"],
                     "ann_id":row["ann_id"],"image_id":image_id,"split":"train",
                     "width":image_info["width"],"height":image_info["height"],
                     "expression":expressions[0],"references":expressions,"target_box":target_box})
    if len(selected)==205:
        break
assert len(selected)==205
calibration, evaluation = selected[:5],selected[5:]
save_json("sample.json",{"dataset":dataset_id,"revision":REVISIONS[dataset_id],"seed":SEED,
                         "partition":"unc","split":"validation","source_rows":len(metadata),
                         "source_sha256":metadata_sha,"calibration":calibration,"evaluation":evaluation})
print("RefCOCO UNC val:",len(metadata),"референтов; выбрано",len(selected),"изображений")

RefCOCO UNC val: 3811 референтов; выбрано 205 изображений


In [8]:
def check_image(record):
    image = coco_image(record,year=2014)
    assert image.size==(record["width"],record["height"])
    x1,y1,x2,y2 = record["target_box"]
    assert -1<=x1<x2<=image.width+1 and -1<=y1<y2<=image.height+1
    return {"question_id":record["question_id"],"image_id":record["image_id"],
            "width":image.width,"height":image.height,
            "rgb_sha256":hashlib.sha256(image.tobytes()).hexdigest()}
with ThreadPoolExecutor(max_workers=8) as pool:
    image_checks = list(pool.map(check_image,calibration+evaluation))
assert len(image_checks)==205
save_json("image_checks.json",image_checks)
print("Проверено оригинальных COCO изображений и эталонных рамок:",len(image_checks))

Проверено оригинальных COCO изображений и эталонных рамок: 205


In [ ]:
model, processor = load_vlm(max_pixels=1000000//784*784)

def grounding(record):
    image = coco_image(record,year=2014)
    prompt = ('Locate the object described as: "'+record["expression"]+'". '
              'Return only JSON {"bbox_2d": [x1, y1, x2, y2]}. '
              'Use absolute pixel coordinates in the resized input image, not normalized coordinates. Return one box.')
    result = vlm_answer(model,processor,image,prompt,96)
    parsed,error = parse_box(result["answer"])
    prediction = None
    if error is None:
        prediction,error = original_box(parsed,image.size,(result["processed_width"],result["processed_height"]))
    return {**record,**result,"original_width":image.width,"original_height":image.height,
            "raw_box":parsed,"predicted_box":prediction,"error":error,"IoU":iou(prediction,record["target_box"])}

calibration_rows = [grounding(record) for record in tqdm(calibration)]
save_json("coordinate_calibration.json",calibration_rows)
fig,axes = plt.subplots(1,5,figsize=(20,5))
for row,ax in zip(calibration_rows,axes):
    ax.imshow(coco_image(row,year=2014))
    draw_box(ax,row["target_box"],"lime","Target")
    draw_box(ax,row["predicted_box"],"red","Prediction")
    ax.set_title(f"IoU={row['IoU']:.3f}\n{row['expression'][:45]}")
    ax.axis("off")
save_figure("coordinate_calibration.png")
display(pd.DataFrame(calibration_rows)[["question_id","raw_box","original_width","original_height","processed_width","processed_height","IoU","error"]])

In [ ]:
calibration_path = OUT/"coordinate_review.json"
if calibration_path.exists():
    coordinate_review = json.loads(calibration_path.read_text(encoding="utf-8"))
else:
    decision = input("После просмотра всех пяти изображений подтверждён формат пикселей resized input? yes/no: ").strip().lower()
    evidence = input("Краткое обоснование по пяти изображениям и размерам: ").strip()
    coordinate_review = {"confirmed":decision=="yes" and bool(evidence),"format":"processed_pixels_xyxy",
                         "evidence":evidence,"question_ids":[r["question_id"] for r in calibration_rows]}
    save_json("coordinate_review.json",coordinate_review)
assert coordinate_review["confirmed"], "Формат не подтверждён; нельзя считать основную оценку корректной."

In [ ]:
answers_path = OUT/"answers.json"
answers = json.loads(answers_path.read_text(encoding="utf-8")) if answers_path.exists() else []
done = {r["question_id"] for r in answers}
for record in tqdm(evaluation,desc="RefCOCO 200"):
    if record["question_id"] not in done:
        answers.append(grounding(record))
        save_json("answers.json",answers)
assert len(answers)==200 and len({r["question_id"] for r in answers})==200
metrics = {"n":len(answers),"mean_IoU":float(np.mean([r["IoU"] for r in answers])),
           "Acc_at_0.5":float(np.mean([r["IoU"]>=0.5 for r in answers])),
           "invalid_rate":float(np.mean([r["error"] is not None for r in answers]))}
save_json("metrics.json",metrics)
display(pd.DataFrame([metrics]))

## 6. Полученные результаты
В метриках участвуют все 200 примеров, включая невалидные ответы. Средний IoU и Acc@0.5 не заменяют семантическую проверку выбранного объекта. Классификация ниже выполняется просмотром каждого валидного предсказания; IoU используется для оценки границ только после подтверждения объекта.

Доля семантических ошибок может отличаться от 1−Acc@0.5: сильно перекрывающийся неверный объект может иметь высокий IoU. Таблица содержит доли от всех 200 примеров и от числа ошибок; неопределённые случаи требуют дополнительного просмотра.

In [ ]:
review_path = OUT/"error_review.json"
reviews = json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else []
reviewed = {r["question_id"] for r in reviews}
for row in answers:
    if row["question_id"] in reviewed:
        continue
    if row["error"] is not None:
        label,reason = "invalid_response",row["error"]
    else:
        fig,ax = plt.subplots(figsize=(7,5))
        ax.imshow(coco_image(row,year=2014))
        draw_box(ax,row["target_box"],"lime","Target")
        draw_box(ax,row["predicted_box"],"red","Prediction")
        ax.set_title(f"{row['expression']} | IoU={row['IoU']:.3f}")
        ax.legend()
        plt.show()
        choice = input("Красная рамка относится к нужному объекту? yes/no/uncertain: ").strip().lower()
        while choice not in {"yes","no","uncertain"}:
            choice = input("Введите yes, no или uncertain: ").strip().lower()
        label = "wrong_object" if choice=="no" else "uncertain" if choice=="uncertain" else "inaccurate_box" if row["IoU"]<0.5 else "success"
        reason = input("Обоснование по изображению: ").strip()
        while not reason:
            reason = input("Введите непустое обоснование: ").strip()
    reviews.append({"question_id":row["question_id"],"label":label,"reason":reason})
    save_json("error_review.json",reviews)
counts = pd.Series(Counter(r["label"] for r in reviews)).rename_axis("type").reset_index(name="count")
counts["fraction_all"] = counts["count"]/200
errors = int(counts.loc[counts.type!="success","count"].sum())
counts["fraction_errors"] = counts.apply(lambda r:r["count"]/errors if r["type"]!="success" and errors else np.nan,axis=1)
counts.to_csv(OUT/"error_types.csv",index=False)
display(counts)
if any(r["label"]=="uncertain" for r in reviews):
    print("Есть неопределённые случаи; полная достоверная классификация трёх типов ошибок не завершена.")

## 7. Анализ и выводы

In [ ]:
label_map = {r["question_id"]:r["label"] for r in reviews}
success = [r for r in answers if r["IoU"]>=0.5 and label_map[r["question_id"]]=="success"][:3]
failure = [r for r in answers if r["IoU"]<0.5 and label_map[r["question_id"]] not in {"success","uncertain"}][:3]
examples = success+failure
fig,axes = plt.subplots(2,3,figsize=(15,10))
for ax in axes.flat:
    ax.axis("off")
for row,ax in zip(examples,axes.flat):
    ax.imshow(coco_image(row,year=2014))
    draw_box(ax,row["target_box"],"lime","Target")
    draw_box(ax,row["predicted_box"],"red","Prediction")
    ax.set_title(f"{row['expression'][:55]}\nIoU={row['IoU']:.3f}; {label_map[row['question_id']]}")
    ax.legend()
save_figure("grounding_examples.png")
save_json("visualized_examples.json",[{"question_id":r["question_id"],"IoU":r["IoU"]} for r in examples])
if len(success)<3 or len(failure)<3:
    print("В фактических данных недостаточно примеров одного типа для схемы 3+3; дополнительные примеры не подставлены.")

## 8. Контрольные вопросы
**Почему Acc@0.5 часто используют вместо среднего IoU?** Порог даёт простую долю успешно локализованных объектов и облегчает сравнение работ. Он менее чувствителен к мелким изменениям хорошей рамки, но теряет информацию о границах около порога. Средний IoU дополняет, а не отменяет такую оценку.

In [9]:
unit = json.loads((OUT/"unit_tests.json").read_text(encoding="utf-8"))
findings = [f"CPU-тесты: IoU совпадающих рамок=1, непересекающихся=0, частичное пересечение={unit['IoU_overlap']:.8f}. Пройдены 100 случайных проверок симметрии, проверки JSON и перевода координат."]
if (OUT/"sample.json").exists():
    findings.append("Подготовлена выборка RefCOCO UNC validation: 200 оценочных и 5 отдельных калибровочных примеров. Проверены исходные аннотации, размеры и контрольные суммы всех 205 чистых изображений COCO.")
if (OUT/"metrics.json").exists():
    table = pd.DataFrame([json.loads((OUT/"metrics.json").read_text(encoding="utf-8"))])
    findings.append("Основная оценка выполнена на 200 примерах отдельно от пяти калибровочных изображений. Невалидные ответы учтены с IoU=0.")
    if (OUT/"error_types.csv").exists():
        findings.append("Классификация: "+pd.read_csv(OUT/"error_types.csv").to_string(index=False))
    else:
        findings.append("Ручная семантическая классификация ошибок не завершена.")
else:
    table = pd.DataFrame([{"test":"identical","IoU":unit["IoU_identical"]},{"test":"disjoint","IoU":unit["IoU_disjoint"]},{"test":"overlap","IoU":unit["IoU_overlap"]}])
    findings += ["Инференс Qwen2.5-VL на пяти калибровочных и 200 оценочных примерах не выполнен: CUDA GPU недоступна. Документированная система координат реализована, но ещё не подтверждена реальными предсказаниями этой модели.",
                 "Средний IoU, Acc@0.5, доли ошибок модели и шесть примеров отсутствуют. Подготовлены загрузка оригиналов без рамок, калибровка, инференс, метрики и ручная проверка выбранного объекта.",
                 "Низкий IoU не позволяет сам по себе отличить неверный объект от неточной рамки. Поэтому тип ошибки не назначается только по числовому порогу."]
write_report("ПЗ-6. Grounding и IoU", "Оценить локализацию по описанию.",
             "Реализованы JSON-парсер, перевод из пикселей resized image, IoU, полный протокол 5+200 и визуальная классификация ошибок.",findings,table,OUT/"grounding_examples.png")

PDF: C:\Users\darri\PycharmProjects\cv061126\PZ_6\report.pdf
